# 20 Comparación de resultados entre AppsScript y GNEWS

Al ejecutar la primera celda, se pide la autorización para acceder a Gdrive. A veces falla en el primer intento, volver a intentarlo.

In [ ]:
import gspread
from google.oauth2.service_account import Credentials

import pandas as pd

url_sin_detenidos = 'https://docs.google.com/spreadsheets/d/1xKAdnEC8HP4b5MJ-HrOGKqa7cXE-ue336RsWkIAhnYg/edit?gid=762130750#gid=762130750'

try:
    from google.colab import auth
    import google.auth

    # Running in Colab
    auth.authenticate_user()
    creds, _ = google.auth.default()
    client = gspread.authorize(creds)

    !pip install gnews
except ImportError:
    from google.oauth2.service_account import Credentials

    # Running locally or outside Colab
    SCOPES = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ]
    creds = Credentials.from_service_account_file(
        "../secrets/credentials.json", scopes=SCOPES
    )
    client = gspread.authorize(creds)

from gnews import GNews

spreadsheet = client.open_by_url(url_sin_detenidos)

### Dataframe con artículos (sin el texto) registrados por el RSS

In [ ]:
# ws = spreadsheet.get_worksheet(0)
ws = spreadsheet.worksheet('INBOX')

# 1. Get raw cell values as a 2D list
raw_values = ws.get_all_values()

# 2. Define your own custom header names
headers = ['Fecha_deteccion', 'Medio', 'Titulo', 'Link', 'Keyword_detectada',
        'Fuente', 'Revisado', 'Validado', 'Observaciones', 'Estado_IA',
        'Palabras_detectadas', 'Puntaje', 'Puntaje_solo_titulo', 'otro_2']

# 3. Map values to records (skipping row 0 if row 0 has the old headers)
all_records = gspread.utils.to_records(headers, raw_values[1:])

# Example output
# for row in all_records[:2]:  # Print first 2 rows
#     print(row)

# {'Fecha_deteccion': 'desde aca ejecuté el nuevo código', 'Medio': '', 'Titulo': '', 'Link': '', 'Keyword_detectada': '', 'Fuente': '', 'Revisado': 'FALSE', 'Validado': '', 'Observaciones': '', 'Estado_IA': '', 'Palabras_detectadas': '', 'Puntaje': '', 'Puntaje_solo_titulo': '', 'otro_2': ''}
# {'Fecha_deteccion': '4/8/2026', 'Medio': 'Google News', 'Titulo': 'Facundo Moyano fue demorado en Belgrano: pelea de pareja y estupefacientes - andigital.com.ar', 'Link': 'https://news.google.com/rss/articles/CBMisAFBVV95cUxOMW5QRFdDeFRrZkVKbGFqTHljUnVMQkVSV1pRazQ4eXo3ajAwR1kzMDRWN2JaY01HOEdpeEp0TmFvQS04Z2VEMDdXQ3AyT29lemwzRHhTUURndUhrRDAtWU0tT3R1N1pmejd6c2hfQ0I5UnV2emlQdmthczZUX0FIbkRpY2NEY0Z0c0k4ZkJUNFhWMWZxdHF3ZTNndVR2ZDRkdU5MSW1PYmItU1kwd0lwUw?oc=5', 'Keyword_detectada': 'demorado', 'Fuente': 'https://news.google.com/rss/search?q=demorado%20CABA&hl=es-419&gl=AR&ceid=AR:es-419', 'Revisado': 'FALSE', 'Validado': 'PENDIENTE', 'Observaciones': '', 'Estado_IA': 'PROCESADO', 'Palabras_detectadas': 'POLICIA: policia, efectivo, uniformado, comisaria, policial, GNA, DIR | VICTIMA: demorado, victima | VIOLENCIA_POLICIAL: operativo | VIOLENCIA_GENERAL: incidente, golpe, violencia | CABA: ciudad de buenos aires, Belgrano, hospital pirovano', 'Puntaje': '12', 'Puntaje_solo_titulo': '', 'otro_2': ''}

# Convert records directly to DataFrame
df_sd = pd.DataFrame(all_records)

# Remove invalid rows: no Title
col = df_sd.columns[2]

# Filter out empty/whitespace strings and NaNs
df_sd = df_sd[df_sd[col].astype(str).str.strip().ne("") & df_sd[col].notna()]

# Reset index, keep the old index for reference, matching the row number in the gdrive sheet.
df_sd.reset_index(names="gdrive_index", inplace=True)

# Remove column "Medio", it's always equal to "Google News"
df_sd.drop(columns=["Medio"], inplace=True)

# Remove column "Fuente", it shows the search source URL, not the url for the article
df_sd.drop(columns=["Fuente"], inplace=True)

# Últimos artículos
df_sd.iloc[-5:]

El RSS devuelve artículos de cualquier fecha, no las últimas noticias. Acceder a algún artículo para verificar que la fecha es de años anteriores.

In [ ]:
print(df_sd.iloc[388]['Link'])

### Dataframe con artículos buscados por GNews (sin el texto)

Palabras de la búsqueda en AppsScript:

/***********************  
   RSS búsquedas seguras  
 ***********************/  
const SEARCH_QUERIES = [  
  'detenido CABA',  
  'detenida CABA',  
  'detuvieron CABA',  
  'demorado CABA',  
  'represion policial CABA',  
  'represión policial CABA',  
  'detenido "Capital Federal"',  
  'represion policial "Capital Federal"',  
  'detenido "Ciudad de Buenos Aires"',  
  'represion policial "Ciudad de Buenos Aires"',  
  'violencia policial "Ciudad de Buenos Aires"',  
  'gatillo facil "Ciudad de Buenos Aires"',  
  'abuso policial"Ciudad de Buenos Aires"',  
  'violencia institucional "Ciudad de Buenos Aires"',  
];  

In [ ]:
# Sitios donde realizar búsquedas por sitio
sites = [
    "www.agenciapacourondo.com.ar",
    "www.c5n.com",
    "www.eldestapeweb.com",
    "www.clarin.com",
    "www.infobae.com",
    "www.laizquierdadiario.com",
    "www.lanacion.com.ar",
    "www.pagina12.com.ar",
    "www.perfil.com",
    "www.tiempoar.com.ar",
    "tn.com.ar",
]

exclude = [
    "www.facebook.com",
    "www.instagram.com",
    "www.laizquierdadiario.cl"
]

keywords = [
    'detenido',  
    'detenida',  
    'detuvieron',  
    'demorado',  
    'represión policial',  
    'represión policial CABA',  
    'detenido "Capital Federal"',  
    'represion policial "Capital Federal"',  
    'detenido "Ciudad de Buenos Aires"',  
    'represion policial "Ciudad de Buenos Aires"',  
    'violencia policial "Ciudad de Buenos Aires"',  
    'gatillo facil "Ciudad de Buenos Aires"',  
    'abuso policial"Ciudad de Buenos Aires"',  
    'violencia institucional "Ciudad de Buenos Aires"',
]

In [ ]:
start = (2026, 3, 20)
end = (2026, 3, 31)

keywords = [
    'represión',
]

all_articles = []

for k in keywords:
    gn = GNews(language="es-419", country="AR")
    gn.start_date = start
    gn.end_date = end

    # Primero una búsqueda por sitio.
    for site in sites:
        articles = gn.get_news(f"{k} site:{site}")
        if len(articles) == 100:
            print(f"Warning: Reached 100 articles for keyword '{k}' on site '{site}'")

        # Handle pygooglenews dictionary output ('entries') or raw list
        entries = articles["entries"] if isinstance(articles, dict) else articles
        all_articles.extend(entries)

    # Ahora una búsqueda general que no esté limitada a un sitio específico.
    gn.exclude_websites = sites + exclude

    articles = gn.get_news(k)
    if len(articles) == 100:
        print(f"Warning: Reached 100 articles for keyword '{k}' in general search")

    entries = articles["entries"] if isinstance(articles, dict) else articles
    all_articles.extend(entries)

# Convert the combined list into a single DataFrame
df = pd.DataFrame(all_articles)
df

In [ ]:
df.loc[df['title'] == df.iloc[1]['title']]

In [ ]:
df_unique = df.value_counts(subset=['title']).reset_index(name='count')
df_unique

In [ ]:
print(df.iloc[2]['url'])


## Observaciones

- Cuando se busca 'gatillo fácil', siempre aparecen artículos de https://www.laizquierdadiario.cl aunque en los artículos no aparezca 'gatillo' ni 'fácil'.
- https://www.laizquierdadiario.cl muestra artículos de Argentina.
- revista crisis, instagram, facebook: ¿excluir?
- country='AR' , Google ignores this